# Task 1: Cosine Similarity Between Two Images

Load two images with Pillow, convert them to grayscale, turn them into NumPy arrays, and compute the cosine similarity between them.

In [ ]:
import numpy as np
from PIL import Image

def load_image_as_array(image_path, size=None):
    """
    Load an image from disk, convert it to grayscale, and return it as a NumPy array.

    Parameters
    ----------
    image_path : str
        Path to the image file.
    size : tuple(int, int), optional
        If given, resizes both images to the same (width, height) so their
        arrays can be compared. Cosine similarity requires equal-length vectors.

    Returns
    -------
    np.ndarray
        2D array of pixel intensities (0-255), dtype float64.
    """
    img = Image.open(image_path).convert("L")  # "L" = 8-bit grayscale
    if size is not None:
        img = img.resize(size)
    return np.asarray(img, dtype=np.float64)


def cosine_similarity(vec_a, vec_b):
    """
    Compute cosine similarity between two vectors (or arrays, which get flattened).

    cosine_similarity = (A . B) / (||A|| * ||B||)
    """
    a = vec_a.flatten()
    b = vec_b.flatten()

    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0  
    
    return dot_product / (norm_a * norm_b)


def compare_images(image_path_1, image_path_2):
    """
    Load two images, convert to grayscale, and return their cosine similarity.
    Automatically resizes the second image to match the first if dimensions differ.
    """
    arr1 = load_image_as_array(image_path_1)
    arr2 = load_image_as_array(image_path_2)

    if arr1.shape != arr2.shape:
        target_size = (arr1.shape[1], arr1.shape[0])
        arr2 = load_image_as_array(image_path_2, size=target_size)

    similarity = cosine_similarity(arr1, arr2)
    return similarity, arr1, arr2


### Demo

Since no real image files are attached, this creates two small sample images on the fly to show the functions working end-to-end. Replace `"image1.png"` / `"image2.png"` with your own file paths to use it for real.

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)

sample1 = Image.fromarray((rng.random((100, 100, 3)) * 255).astype(np.uint8))
sample1.save("image1.png")
noisy = np.asarray(sample1, dtype=np.float64) + rng.normal(0, 25, (100, 100, 3))
noisy = np.clip(noisy, 0, 255).astype(np.uint8)
sample2 = Image.fromarray(noisy)
sample2.save("image2.png")

similarity, arr1, arr2 = compare_images("image1.png", "image2.png")
print(f"Cosine similarity between image1 and image2: {similarity:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(arr1, cmap="gray")
axes[0].set_title("Image 1 (grayscale)")
axes[0].axis("off")
axes[1].imshow(arr2, cmap="gray")
axes[1].set_title("Image 2 (grayscale)")
axes[1].axis("off")
plt.tight_layout()
plt.show()


---
# Task 2: Two Matplotlib Plots

1. **Damped cosine wave**
2. **Linear chirp** a cosine whose frequency increases linearly from f = 3 Hz to f = 10 Hz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 2, 2000)

### Plot 1: Damped Cosine

$$x(t) = e^{-\alpha t} \cos(2\pi f t)$$

In [ ]:
alpha = 2.0
f0 = 5.0

damped_cosine = np.exp(-alpha * t) * np.cos(2 * np.pi * f0 * t)

plt.figure(figsize=(9, 4))
plt.plot(t, damped_cosine, color="steelblue")
plt.plot(t, np.exp(-alpha * t), "--", color="gray", linewidth=1, label="envelope")
plt.plot(t, -np.exp(-alpha * t), "--", color="gray", linewidth=1)
plt.title("Damped Cosine Wave")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Plot 2: Linear Chirp (frequency sweeps from f = 3 Hz to f = 10 Hz)

For a linear chirp, the instantaneous frequency changes linearly with time:

$$f(t) = f_0 + k t, \qquad k = \frac{f_1 - f_0}{T}$$

The phase is the integral of $2\pi f(t)$, giving:

$$x(t) = \cos\!\Big(2\pi \big(f_0 t + \tfrac{k}{2} t^2\big)\Big)$$

In [ ]:
f0 = 3.0
f1 = 10.0
T = t[-1]
k = (f1 - f0) / T

linear_chirp = np.cos(2 * np.pi * (f0 * t + 0.5 * k * t**2))

plt.figure(figsize=(9, 4))
plt.plot(t, linear_chirp, color="darkorange")
plt.title("Linear Chirp (f = 3 Hz \u2192 10 Hz)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()